## עמודה B: agent_code
קוד הסוכן שמכר את הפוליסה. הסוכן ידוע ביום החתימה ולכן אין כאן דליפת מידע.
נבדוק את אחידות הפורמט, את מספר הסוכנים, את כמות הנהגים לכל סוכן ואת הקשר לעלות התביעות.

In [6]:
# ייבוא ספריות טעינת הדאטה וקביעת מצב אקראי קבוע
import pandas as pd
import numpy as np

RANDOM_STATE = 42  # Required by the assignment
df = pd.read_csv("dataset.csv")
print(df.shape)

(45000, 30)


## עמודה A: customer_id_hash
מזהה לקוח מוצפן. לכל נהג יש קוד ייחודי, ולכן אין בעמודה מידע שיכול לעזור בחיזוי, ונהג חדש יגיע תמיד עם קוד שהמודל לא ראה. העמודה תוסר.

In [13]:
# בדיקה שכל מזהה לקוח מופיע פעם אחת בלבד
print("Unique IDs:", df["customer_id_hash"].is_unique)

Unique IDs: True


In [7]:
# בדיקת ערכים חסרים ובדיקה שכל הקודים באותו פורמט של אות ושלוש ספרות
print("Missing:", df["agent_code"].isna().sum())
print("Valid format:", df["agent_code"].str.match(r"^A\d{3}$").mean())

Missing: 0
Valid format: 1.0


In [8]:
# כמה סוכנים יש וכמה נהגים יש לכל סוכן
counts = df["agent_code"].value_counts()
print("Number of agents:", counts.size)
counts.describe()

Number of agents: 300


count     300.000000
mean      150.000000
std       235.319423
min        35.000000
25%        64.000000
50%        87.000000
75%       140.250000
max      2791.000000
Name: count, dtype: float64

In [9]:
# עלות ממוצעת לפי סוכן אצל הסוכנים הקטנים לעומת הגדולים
agent_stats = df.groupby("agent_code")["annual_claim_cost"].agg(["mean", "count"])
print("Small agents under 100 drivers")
print(agent_stats[agent_stats["count"] < 100]["mean"].describe().round(0))
print("\nLarge agents with 200 drivers or more")
print(agent_stats[agent_stats["count"] >= 200]["mean"].describe().round(0))

Small agents under 100 drivers
count     179.0
mean     3016.0
std       955.0
min       726.0
25%      2370.0
50%      2864.0
75%      3605.0
max      6641.0
Name: mean, dtype: float64

Large agents with 200 drivers or more
count      45.0
mean     3013.0
std       426.0
min      2135.0
25%      2764.0
50%      3048.0
75%      3218.0
max      4070.0
Name: mean, dtype: float64


In [10]:
# בדיקה האם כל סוכן עובד במדינה אחת או בכמה מדינות
df.groupby("agent_code")["country"].nunique().describe()

count    300.000000
mean       9.896667
std        0.315681
min        8.000000
25%       10.000000
50%       10.000000
75%       10.000000
max       10.000000
Name: country, dtype: float64

In [12]:
# בדיקת ערבוב: מערבבים את העלויות בין הנהגים ובודקים אם הפיזור האמיתי בין הסוכנים גדול ממה שמקריות יוצרת
rng = np.random.default_rng(RANDOM_STATE)
y = df["annual_claim_cost"].values
agents = df["agent_code"].values

real_spread = pd.Series(y).groupby(agents).mean().std()
shuffled_spreads = [pd.Series(rng.permutation(y)).groupby(agents).mean().std() for _ in range(300)]

print("Real spread between agents:     ", round(real_spread))
print("Average spread after shuffling: ", round(np.mean(shuffled_spreads)))
print("Share of shuffles with larger spread:", round(np.mean(np.array(shuffled_spreads) >= real_spread), 2))

Real spread between agents:      807
Average spread after shuffling:  803
Share of shuffles with larger spread: 0.44


### מסקנה לגבי agent_code
העמודה נקייה, ללא חסרים ובפורמט אחיד. יש 300 סוכנים, רובם קטנים, וכמעט כל סוכן עובד בכל המדינות.
בבדיקת ערבוב, הפיזור האמיתי בין הסוכנים דומה לפיזור שמתקבל כשמערבבים את העלויות באקראי. כלומר, ההבדלים בין הסוכנים מוסברים במקריות.
לכן העמודה תוסר. הוספתה הייתה מוסיפה למודל 300 עמודות ללא מידע ומגדילה את הסיכון ל-overfitting.

In [ ]:
# רשימת העמודות מהחלק שלי שיוסרו מהמודל
DROP_COLS_DRIVER = ["customer_id_hash", "agent_code"]

## עמודה C: country
מדינת הפוליסה. זו עמודה קטגוריאלית עם 10 מדינות. המדינה ידועה ביום החתימה ולכן אין כאן דליפת מידע.
נבדוק את אחידות הערכים, את כמות הנהגים בכל מדינה ואת הקשר בין המדינה לעלות התביעות.

In [14]:
# בדיקת ערכים חסרים ורשימת המדינות עם מספר הנהגים בכל אחת
print("Missing:", df["country"].isna().sum())
df["country"].value_counts()

Missing: 0


country
US    9907
DE    5789
GB    5394
FR    4536
IT    4508
IL    4455
ES    3739
BR    3156
PL    2181
JP    1335
Name: count, dtype: int64

In [15]:
# השוואת עלות התביעות בין המדינות לפי שלושה מדדים
country_stats = df.groupby("country")["annual_claim_cost"].agg(
    avg_cost="mean",
    share_with_claim=lambda s: (s > 0).mean(),
    share_expensive=lambda s: (s > 10000).mean(),
    drivers="count",
).round(3).sort_values("avg_cost")
country_stats

,avg_cost,share_with_claim,share_expensive,drivers
country,,,,
JP,973.666,0.196,0.022,1335
BR,1197.146,0.412,0.013,3156
PL,1247.617,0.379,0.017,2181
ES,1625.003,0.347,0.031,3739
GB,1976.434,0.266,0.059,5394
IT,2049.465,0.350,0.053,4508
FR,2130.829,0.301,0.058,4536
DE,2819.795,0.297,0.093,5789
IL,2993.003,0.369,0.090,4455


### מסקנה לגבי country
העמודה נקייה, עם 10 מדינות וללא ערכים חסרים. יש הבדל גדול ועקבי בעלות בין המדינות, פי 6.7 בין יפן לארה"ב, ובכל מדינה יש אלפי נהגים ולכן ההבדל אינו מקרי. העמודה תישאר.
יש מדינות עם הרבה תביעות קטנות, כמו ברזיל, ומדינות עם תביעות גדולות, כמו ארה"ב.
בפונקציית הניקוי נוסיף חוק שמאחד כתיבה, כדי להגן מפני וריאנטים בקובץ הבדיקה. את המרת המדינה לעמודות 0 ו-1 נבצע בתוך ה-Pipeline.

## עמודה D: area_type
סוג אזור המגורים: עיר, פרבר או כפר. זו עמודה קטגוריאלית שידועה ביום החתימה.
נבדוק את אחידות הערכים, את הקשר לעלות התביעות ואת הקשר לעמודת צפיפות האוכלוסייה.

In [21]:
# בדיקת ערכים חסרים ורשימת סוגי האזורים עם מספר הנהגים בכל אחד
print("Missing:", df["area_type"].isna().sum())
df["area_type"].value_counts()

Missing: 0


area_type
urban       18329
suburban    18086
rural        8585
Name: count, dtype: int64

In [22]:
# השוואת עלות התביעות בין סוגי האזורים לפי שלושה מדדים
df.groupby("area_type")["annual_claim_cost"].agg(
    avg_cost="mean",
    share_with_claim=lambda s: (s > 0).mean(),
    share_expensive=lambda s: (s > 10000).mean(),
    drivers="count",
).round(3)

,avg_cost,share_with_claim,share_expensive,drivers
area_type,,,,
rural,3751.261,0.297,0.121,8585
suburban,3062.542,0.344,0.095,18086
urban,2711.231,0.383,0.077,18329


In [23]:
# בדיקה האם סוג האזור מתאים לצפיפות האוכלוסייה שרשומה לנהג
df.groupby("area_type")["region_density"].describe().round(0)

,count,mean,std,min,25%,50%,75%,max
area_type,,,,,,,,
rural,8585.0,74.0,59.0,4.0,35.0,58.0,93.0,602.0
suburban,18086.0,991.0,563.0,121.0,611.0,863.0,1227.0,8205.0
urban,18329.0,3619.0,2413.0,299.0,1970.0,2997.0,4541.0,20000.0


### מסקנה לגבי area_type
העמודה נקייה, עם שלושה ערכים וללא חסרים. בעיר יש יותר תביעות אך קטנות יותר, ובכפר פחות תביעות אך יקרות יותר, ולכן העמודה משפיעה על העלות ותישאר.
הצפיפות מתאימה לסוג האזור ברוב המקרים. החפיפה הקטנה בקצוות נובעת כנראה מהגדרות שונות בין מדינות ואינה שגיאה.
בפונקציית הניקוי נוסיף חוק שמאחד כתיבה. את המרת האזור לעמודות 0 ו-1 נבצע בתוך ה-Pipeline.

In [24]:
# פונקציית הניקוי של עמודות הנהג שנוסיף לה חוק אחרי כל עמודה שנבדוק
def clean_driver_cols(df):
    df = df.copy()

    # מדינה: הסרת רווחים והמרה לאותיות גדולות
    df["country"] = df["country"].str.strip().str.upper()

    # סוג אזור: הסרת רווחים והמרה לאותיות קטנות
    df["area_type"] = df["area_type"].str.strip().str.lower()

    return df

# בדיקה שהפונקציה רצה על כל הדאטה ועל שורה בודדת
print(clean_driver_cols(df).shape)
print(clean_driver_cols(df.iloc[[0]]).shape)

(45000, 30)
(1, 30)
